In [7]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os, json, time
from pathlib import Path

load_dotenv(override=True)

True

In [8]:
# Xóa DB cũ nếu có
if Path("donhang.db").exists():
    Path("donhang.db").unlink()
    print("Đã xóa database cũ\n")

from donhang import QuanLyDonHang

# Tạo dữ liệu mẫu
d1 = QuanLyDonHang.tao_don("Nguyen Van A", "Áo thun basic",  2, 150_000, "Giao sáng")
time.sleep(1)
d2 = QuanLyDonHang.tao_don("Tran Thi B",   "Quần jean slim", 1, 350_000, "")
time.sleep(1)
d3 = QuanLyDonHang.tao_don("Le Van C",     "Giày sneaker",   1, 850_000, "Size 42")

print(f"  Đơn 1: {d1['ma_don']} — {d1['tong_tien']:,.0f}đ")
print(f"  Đơn 2: {d2['ma_don']} — {d2['tong_tien']:,.0f}đ")
print(f"  Đơn 3: {d3['ma_don']} — {d3['tong_tien']:,.0f}đ")

QuanLyDonHang.cap_nhat_trang_thai(d2["ma_don"], "dang_xu_ly")
QuanLyDonHang.cap_nhat_trang_thai(d3["ma_don"], "da_giao")

dt = QuanLyDonHang.doanh_thu()
print(f"\nDoanh thu: {dt['tong_doanh_thu']:,.0f}đ | {dt['so_don_hang']} đơn")

# Lưu mã đơn để dùng trong các cells sau
MA_DON_1, MA_DON_2, MA_DON_3 = d1["ma_don"], d2["ma_don"], d3["ma_don"]

Đã xóa database cũ

  Đơn 1: DH20260406101932 — 300,000đ
  Đơn 2: DH20260406101933 — 350,000đ
  Đơn 3: DH20260406101934 — 850,000đ

Doanh thu: 1,500,000đ | 3 đơn


In [9]:
server_params ={
    "command": "python",
    "args": ["donhang_server.py"]
}

async with MCPServerStdio(params=server_params, client_session_timeout_seconds=15) as server:
    mcp_tools = await server.list_tools()

print(f"MCP Server: quan_ly_don_hang | {len(mcp_tools)} tools\n")
for t in mcp_tools:
    print(f"-> {t.name}")
    print(f" {t.description.split(chr(10))[0]}\n")

MCP Server: quan_ly_don_hang | 5 tools

-> tao_don_hang
 Tạo một đơn hàng mới cho khách hàng.

-> cap_nhat_trang_thai_don
 Cập nhật trạng thái xử lý của một đơn hàng.

-> xem_chi_tiet_don
 Xem thông tin chi tiết của một đơn hàng theo mã đơn.

-> danh_sach_don_hang
 Lấy danh sách đơn hàng, có thể lọc theo khách hàng hoặc trạng thái.

-> bao_cao_doanh_thu
 Tính tổng doanh thu và số đơn hàng trong khoảng thời gian.



In [7]:
# xem json schema
tool_tao = next(t for t in mcp_tools if t.name == 'tao_don_hang')

print(json.dumps(tool_tao.inputSchema, indent=2, ensure_ascii=False))

{
  "properties": {
    "khach_hang": {
      "title": "Khach Hang",
      "type": "string"
    },
    "san_pham": {
      "title": "San Pham",
      "type": "string"
    },
    "so_luong": {
      "title": "So Luong",
      "type": "integer"
    },
    "don_gia": {
      "title": "Don Gia",
      "type": "number"
    },
    "ghi_chu": {
      "default": "",
      "title": "Ghi Chu",
      "type": "string"
    }
  },
  "required": [
    "khach_hang",
    "san_pham",
    "so_luong",
    "don_gia"
  ],
  "title": "tao_don_hangArguments",
  "type": "object"
}


In [10]:
# Agent
instructions = """
Bạn là trợ lý quản lý đơn hàng cho một shop thời trang online Việt Nam.

Nhiệm vụ: Giúp chủ shop quản lý đơn hàng — tạo mới, cập nhật trạng thái,
tra cứu, và báo cáo doanh thu.

Quy tắc:
- Xác nhận mã đơn sau khi tạo
- Format tiền tệ theo VNĐ có dấu phân cách nghìn
- Trả lời bằng tiếng Việt, ngắn gọn và rõ ràng
"""

request = f"""
Xử lý các việc sau:

1. Xem chi tiết đơn {MA_DON_1} của Nguyen Van A
2. Cập nhật đơn {MA_DON_1} sang trạng thái 'dang_xu_ly'
3. Tạo đơn mới:
   - Khách: Pham Thi D
   - Sản phẩm: Túi xách da
   - Số lượng: 1 | Đơn giá: 520.000đ
   - Ghi chú: Khách yêu cầu gói quà
4. Báo cáo tổng doanh thu hiện tại

Thực hiện lần lượt và xác nhận từng bước.
"""

async with MCPServerStdio(params=server_params, client_session_timeout_seconds=15) as mcp_server:
    agent = Agent(
        name="quan-ly-don-hang",
        instructions=instructions,
        model="gpt-4o-mini",
        mcp_servers=[mcp_server]
    )
    with trace("quan-ly-don-hang-ngay3"):
        result = await Runner.run(agent, request, max_turns=15)
        print(result.final_output)

1. **Chi tiết đơn DH20260406101932**:
   - Khách hàng: Nguyen Van A
   - Sản phẩm: Áo thun basic
   - Số lượng: 2
   - Đơn giá: 150.000đ
   - Trạng thái: Chờ xác nhận
   - Ghi chú: Giao sáng
   - Ngày tạo: 2026-04-06

2. **Cập nhật trạng thái**: Đơn đã được cập nhật sang trạng thái **"đang xử lý"**.

3. **Đơn hàng mới** đã được tạo thành công:
   - Mã đơn: **DH20260406102010**
   - Khách hàng: Pham Thi D
   - Sản phẩm: Túi xách da
   - Số lượng: 1
   - Đơn giá: 520.000đ
   - Trạng thái: Chờ xác nhận
   - Ghi chú: Khách yêu cầu gói quà

4. **Tổng doanh thu hiện tại**:
   - **Tổng doanh thu**: 1.500.000đ
   - **Số đơn hàng**: 3

Nếu bạn cần thêm thông tin gì khác, hãy cho tôi biết!


In [11]:
# Sau khi agent chạy, verify bằng Python trực tiếp
import sqlite3

conn = sqlite3.connect("donhang.db")
cursor = conn.cursor()
cursor.execute("SELECT id, khach_hang, san_pham, trang_thai FROM don_hang ORDER BY ngay_tao DESC")
rows = cursor.fetchall()
conn.close()

print(f"Database: {len(rows)} đơn hàng\n")
print(f"{'Mã đơn':<20} {'Khách hàng':<16} {'Sản phẩm':<18} {'Trạng thái'}")
print("─" * 72)
for row in rows:
    print(f"{row[0]:<20} {row[1]:<16} {row[2]:<18} {row[3]}")

# Verify đơn 1 đã được cập nhật
don_1 = QuanLyDonHang.xem_don(MA_DON_1)
expected = "dang_xu_ly"
actual = don_1.get("trang_thai", "?")
icon = "✓" if actual == expected else "⚠"
print(f"\n{icon} {MA_DON_1}: {actual} (kỳ vọng: {expected})")

Database: 4 đơn hàng

Mã đơn               Khách hàng       Sản phẩm           Trạng thái
────────────────────────────────────────────────────────────────────────
DH20260406102010     Pham Thi D       Túi xách da        cho_xac_nhan
DH20260406101934     Le Van C         Giày sneaker       da_giao
DH20260406101933     Tran Thi B       Quần jean slim     dang_xu_ly
DH20260406101932     Nguyen Van A     Áo thun basic      dang_xu_ly

✓ DH20260406101932: dang_xu_ly (kỳ vọng: dang_xu_ly)


# MCP Client

In [12]:
from donhang_client import liet_ke_tools, goi_tool, doc_resource_khach, lay_tools_openai

tools_raw = await liet_ke_tools()
print(f"Nhận được {len(tools_raw)} tools:")
for t in tools_raw:
    print(f"- {t.name}: {t.description.split(chr(10))[0]}")

print("\n Convert sang OpenAI format --\n")
openai_tools = await lay_tools_openai()
for t in openai_tools:
    print(f"- {t.name} [{type(t).__name__}]")

Nhận được 5 tools:
- tao_don_hang: Tạo một đơn hàng mới cho khách hàng.
- cap_nhat_trang_thai_don: Cập nhật trạng thái xử lý của một đơn hàng.
- xem_chi_tiet_don: Xem thông tin chi tiết của một đơn hàng theo mã đơn.
- danh_sach_don_hang: Lấy danh sách đơn hàng, có thể lọc theo khách hàng hoặc trạng thái.
- bao_cao_doanh_thu: Tính tổng doanh thu và số đơn hàng trong khoảng thời gian.

 Convert sang OpenAI format --

- tao_don_hang [FunctionTool]
- cap_nhat_trang_thai_don [FunctionTool]
- xem_chi_tiet_don [FunctionTool]
- danh_sach_don_hang [FunctionTool]
- bao_cao_doanh_thu [FunctionTool]


In [13]:
# goi tool thu cong
result = await goi_tool("tao_don_hang", {
    "khach_hang": "Hoang Van E",
    "san_pham":   "Kính mát UV400",
    "so_luong":   1,
    "don_gia":    280_000,
    "ghi_chu":    "Giao COD"
})

raw_text = result.content[0].text
print(f"Raw response từ server:\n{raw_text}\n")

don = json.loads(raw_text)
print(f"Parsed: {don['ma_don']} — {don['tong_tien']:,.0f}đ — {don['trang_thai']}")

Raw response từ server:
{
  "ma_don": "DH20260406102035",
  "tong_tien": 280000.0,
  "trang_thai": "cho_xac_nhan"
}

Parsed: DH20260406102035 — 280,000đ — cho_xac_nhan


In [15]:
# Dung tools openai truc tiep voi Agent (khong qua mcp_servers)
request_2 = f"""
Kiểm tra đơn hàng {MA_DON_3} của Le Van C và cho biết trạng thái.
Sau đó báo cáo tổng doanh thu hiện tại.
"""

agent_2 = Agent(
    name         = "quan-ly-don-hang-v2",
    instructions = instructions,
    model        = "gpt-4o-mini",
    tools        = openai_tools,  # tools thay vì mcp_servers
)

with trace("quan-ly-don-hang-client-thu-cong"):
    result_2 = await Runner.run(agent_2, request_2, max_turns=8)
    print(result_2.final_output)

print("\n── Hai cách, cùng kết quả ──────────────────────────────────")
print("  mcp_servers: SDK tự quản lý lifecycle server")
print("  tools      : Bạn tự quản lý lifecycle server")
print("  Production → dùng mcp_servers")
print("  Framework khác không hỗ trợ MCP → dùng tools")

Đơn hàng **DH20260406101934** của khách hàng **Le Van C** có trạng thái **đã giao**.

Tổng doanh thu hiện tại là **2.300.000 VNĐ** với tổng số **5 đơn hàng**.

── Hai cách, cùng kết quả ──────────────────────────────────
  mcp_servers: SDK tự quản lý lifecycle server
  tools      : Bạn tự quản lý lifecycle server
  Production → dùng mcp_servers
  Framework khác không hỗ trợ MCP → dùng tools


In [16]:
print("So sánh 3 cách expose Python function cho agent:\n")

print("── 1. @function_tool (simple, internal use only) ─────────────")
print("""
  from agents import function_tool

  @function_tool
  def tinh_tong(so_luong: int, don_gia: float) -> float:
      \"\"\"Tính tổng tiền đơn hàng.\"\"\"
      return so_luong * don_gia

  agent = Agent(..., tools=[tinh_tong])
""")

print("── 2. MCP Server + mcp_servers (sharable, protocol) ──────────")
print("""
  # donhang_server.py
  @mcp.tool()
  async def tao_don_hang(...) -> dict: ...

  # Notebook
  async with MCPServerStdio(params=...) as mcp_server:
      agent = Agent(..., mcp_servers=[mcp_server])
""")

print("── 3. MCP Server + Client thủ công (maximum control) ─────────")
print("""
  tools = await lay_tools_openai()
  agent = Agent(..., tools=tools)
""")

print("── Khi nào dùng gì ───────────────────────────────────────────")
print("  Cách 1: Internal tools, project cá nhân, team nhỏ")
print("  Cách 2: Share tools, production, nhiều consumers")
print("  Cách 3: Framework khác, không có MCP support, education")

So sánh 3 cách expose Python function cho agent:

── 1. @function_tool (simple, internal use only) ─────────────

  from agents import function_tool

  @function_tool
  def tinh_tong(so_luong: int, don_gia: float) -> float:
      """Tính tổng tiền đơn hàng."""
      return so_luong * don_gia

  agent = Agent(..., tools=[tinh_tong])

── 2. MCP Server + mcp_servers (sharable, protocol) ──────────

  # donhang_server.py
  @mcp.tool()
  async def tao_don_hang(...) -> dict: ...

  # Notebook
  async with MCPServerStdio(params=...) as mcp_server:
      agent = Agent(..., mcp_servers=[mcp_server])

── 3. MCP Server + Client thủ công (maximum control) ─────────

  tools = await lay_tools_openai()
  agent = Agent(..., tools=tools)

── Khi nào dùng gì ───────────────────────────────────────────
  Cách 1: Internal tools, project cá nhân, team nhỏ
  Cách 2: Share tools, production, nhiều consumers
  Cách 3: Framework khác, không có MCP support, education
